### **Przykład implementacji sieci ResNet do klasyfikacji obrazów ze zbioru CIFAR-10**

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

#### --- POJEDYNCZY BLOK RESIDUALNY ---

In [3]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(ResidualBlock, self).__init__()

        # Pierwszy splot w bloku (może zmniejszać rozdzielczość, jeśli stride > 1)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        # Drugi splot (zawsze zachowuje rozmiar, stride=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Mechanizm wyrównujący wymiary (np. gdy zwiększamy liczbę kanałów)
        self.downsample = downsample

    def forward(self, x):
        # 1. Zapisujemy autostradę
        skrot = x
        if self.downsample is not None:
            skrot = self.downsample(x)

        # 2. Główna ścieżka transformacji
        out = self.conv1(x)
        out = self.bn1(out)
        out = F.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # 3. Magia ResNetu: Dodajemy autostradę do przetworzonego sygnału
        out += skrot
        out = F.relu(out) # Aktywacja ZAWSZE po dodaniu skrótu

        return out

#### --- MIKROARCHITEKTURA ResNet ---

In [4]:
class ResNetCIFAR(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNetCIFAR, self).__init__()
        self.in_channels = 64

        # 1. Zmodyfikowany Pień Sieci (Delikatny start dla obrazków 32x32)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        # 2. Fabryka bloków (Tworzymy 4 główne etapy)
        # Każdy kolejny etap z wyjątkiem pierwszego zmniejsza obraz 2x i podwaja kanały
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)

        # 3. Klasyfikator końcowy
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1)) # Zgniata mapy cech do 1 piksela
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        downsample = None
        # Jeśli zmieniamy wymiary przestrzenne lub liczbę kanałów, musimy wyrównać skrót
        if stride != 1 or self.in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

        layers = []
        # Pierwszy blok w warstwie może mieć stride=2
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels

        # Pozostałe bloki w warstwie mają już stride=1
        for _ in range(1, num_blocks):
            layers.append(block(out_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1) # Spłaszczamy, pomijając wymiar batcha
        x = self.fc(x)
        return x

# Funkcja pomocnicza do tworzenia ResNet-18
def resnet18_cifar():
    # Zlecamy po 2 bloki w każdej z 4 warstw
    return ResNetCIFAR(ResidualBlock, [2, 2, 2, 2])

#### --- Trening za zbiorze CIFAR-10 ---

In [5]:
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim

In [6]:
# ==========================================
# 1. Przygotowanie danych (Preprocessing)
# ==========================================
# Transformacje dla obrazków CIFAR-10
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(), # Prosta augmentacja
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Pobieranie paczek treningowych
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

100%|██████████| 170M/170M [00:08<00:00, 19.5MB/s]


In [7]:
# ==========================================
# 2. Inicjalizacja modelu, GPU i optymalizatora
# ==========================================
# Wybór urządzenia (GPU jeśli dostępne, inaczej CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używamy urządzenia: {device}")

# Tworzymy naszą sieć ResNet-18 i wysyłamy na kartę graficzną
model = resnet18_cifar().to(device)

# Ustawiamy funkcję straty i optymalizator Adam
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

Używamy urządzenia: cuda


In [8]:
# ==========================================
# 3. Pętla treningowa
# ==========================================
epochs = 5 # Ustawione na 5 dla celów demonstracyjnych

print("Rozpoczynamy trening ResNet-18 na CIFAR-10...")
for epoch in range(epochs):
    model.train() # Ustawiamy model w tryb treningu (aktywuje to np. BatchNorm)
    running_loss = 0.0
    correct = 0
    total = 0

    for i, data in enumerate(trainloader, 0):
        # Pobieramy paczkę danych i wyrzucamy na GPU
        inputs, labels = data[0].to(device), data[1].to(device)

        # Wyzerowanie gradientów
        optimizer.zero_grad()

        # Propagacja w przód (Forward pass)
        outputs = model(inputs)

        # Obliczenie błędu
        loss = criterion(outputs, labels)

        # Propagacja wsteczna (Backward pass) i aktualizacja wag
        loss.backward()
        optimizer.step()

        # Statystyki do wyświetlania
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    # Wyświetlanie wyników po każdej epoce
    epoch_loss = running_loss / len(trainloader)
    epoch_acc = 100. * correct / total
    print(f"Epoka [{epoch+1}/{epochs}] | Błąd (Loss): {epoch_loss:.4f} | Dokładność (Accuracy): {epoch_acc:.2f}%")

print("Trening zakończony pomyślnie!")

Rozpoczynamy trening ResNet-18 na CIFAR-10...
Epoka [1/5] | Błąd (Loss): 1.4926 | Dokładność (Accuracy): 45.09%
Epoka [2/5] | Błąd (Loss): 0.9979 | Dokładność (Accuracy): 64.54%
Epoka [3/5] | Błąd (Loss): 0.7853 | Dokładność (Accuracy): 72.39%
Epoka [4/5] | Błąd (Loss): 0.6398 | Dokładność (Accuracy): 77.60%
Epoka [5/5] | Błąd (Loss): 0.5556 | Dokładność (Accuracy): 80.67%
Trening zakończony pomyślnie!
